# S04 · KV quantization: per-token vs KIVI vs TurboQuant

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 11. Sources: TurboQuant (arXiv:2504.19874; most-starred implementation github.com/0xSero/turboquant, GPL-3.0, linked only); KIVI (arXiv:2402.02750, github.com/jy-yuan/KIVI).

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Compare three ways to quantize keys that have outlier channels: naive per-token groups, KIVI's
per-channel key groups, and TurboQuant's random rotation plus a fixed Gaussian codebook.

## Theory and equations
Group quantization error $\le \Delta/2$ with $\Delta = (\max-\min)/(2^b-1)$, so an outlier in a group hurts every element.
TurboQuant: $y = \sqrt{d}\,R\,x/\|x\|$ has nearly i.i.d. $\mathcal N(0,1)$ coordinates for a Haar-random $R$;
quantize each with the Lloyd-Max codebook; store $\|x\|$. Expected relative error $\approx \sqrt{D_b}$
($D_1 = 0.363$, $D_2 = 0.118$, $D_3 = 0.0345$, $D_4 = 0.0095$).

## Diagram
```
per-token:    [ outlier ch | normal ch ... ]  one range per token  -> range blown up by the outlier
KIVI:         per channel across tokens         -> the outlier channel gets its own range
TurboQuant:   rotate -> outlier energy spread over all coords -> fixed N(0,1) codebook
```

## Implementation

In [ ]:
from inference_lab import quant
k, v = quant.synthetic_kv()                          # 512 tokens, d=128, 4 outlier channels
print("Lloyd-Max N(0,1) 2-bit codebook:", quant.lloyd_max_gaussian(2))

## Experiment

In [ ]:
q = torch.randn(16, 128, generator=torch.Generator().manual_seed(9))
full = quant.attention_out(q, k, v)
e = quant.rel_error
print(f"{'bits':>4} {'K tok':>7} {'K KIVI':>7} {'K TQ':>7} | {'out tok':>8} {'out KIVI':>9} {'out TQ':>7}")
for b in (4, 3, 2):
    kt, kc, ktq = quant.quantize_grouped(k, b, "per_token"), quant.quantize_grouped(k, b, "per_channel"), quant.turboquant(k, b)
    vt, vtq = quant.quantize_grouped(v, b, "per_token"), quant.turboquant(v, b)
    kk, vv = quant.kivi(k, v, b)
    print(f"{b:4d} {e(kt,k):7.3f} {e(kc,k):7.3f} {e(ktq,k):7.3f} | {e(quant.attention_out(q,kt,vt),full):8.3f} "
          f"{e(quant.attention_out(q,kk,vv),full):9.3f} {e(quant.attention_out(q,ktq,vtq),full):7.3f}")
print("bytes/elem at 2 bits: group-32 =", quant.bytes_per_elem(2, 32), " TurboQuant d=128 =", quant.turboquant_bytes_per_elem(2, 128))

## Results: sensitivity to outlier strength

In [ ]:
scales = [0, 2, 5, 10, 15, 25]
rows = {"per-token": [], "KIVI per-channel": [], "TurboQuant": []}
for s_ in scales:
    kx, _ = quant.synthetic_kv(outlier_scale=s_)
    rows["per-token"].append(quant.rel_error(quant.quantize_grouped(kx, 3, "per_token"), kx))
    rows["KIVI per-channel"].append(quant.rel_error(quant.quantize_grouped(kx, 3, "per_channel"), kx))
    rows["TurboQuant"].append(quant.rel_error(quant.turboquant(kx, 3), kx))
for (name, ys), col in zip(rows.items(), [C["waste"], C["memory"], C["compute"]]):
    plt.plot(scales, ys, "o-", color=col, label=name)
plt.xlabel("outlier channel offset (sigma)"); plt.ylabel("3-bit key relative error"); plt.legend(); plt.show()

## Interpretation
Per-token error grows with outlier strength. KIVI isolates outliers by grouping per channel.
TurboQuant removes the dependence differently: after rotation every vector looks Gaussian, so
one fixed codebook is near-optimal and its error stays near the Lloyd-Max value. The rotation
must be drawn independently of the data; a correlated "random" matrix breaks the Gaussian assumption.

## Limitations
Synthetic data; the paper's QJL stage for unbiased inner products is not implemented.

## Conclusion
Outliers decide KV quantization quality. Isolate them (KIVI) or spread them (TurboQuant).